# Customer Churn Prediction Using Machine Learning

**University of Lahore — Artificial Intelligence**

This notebook implements the complete assignment pipeline using the supplied Telco Customer Churn dataset.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
pd.set_option("display.max_columns", None)

## 1. Data Loading and Understanding

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("Data Type"))
display(df.describe(include="all").T)

## 2. Data Preprocessing

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing Values"))
print("Duplicate rows:", df.duplicated().sum())
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop_duplicates().dropna(subset=["Churn"]).copy()
df = df.drop(columns=["customerID"], errors="ignore")
print("Final shape:", df.shape)

## 3. Exploratory Data Analysis

In [ ]:
display(df["Churn"].value_counts().to_frame("Customers"))
print(f"Overall churn rate: {df['Churn'].eq('Yes').mean()*100:.2f}%")
df["Churn"].value_counts().plot(kind="bar"); plt.title("Customer Churn Distribution"); plt.xlabel("Churn"); plt.ylabel("Customers"); plt.xticks(rotation=0); plt.show()

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index")*100
display(contract_churn.round(2))
contract_churn.plot(kind="bar"); plt.title("Churn Percentage by Contract Type"); plt.ylabel("Percentage"); plt.xticks(rotation=0); plt.show()

In [ ]:
for label in ["No","Yes"]:
    plt.hist(df.loc[df["Churn"]==label,"tenure"], bins=25, alpha=.6, label=label)
plt.title("Tenure Distribution by Churn"); plt.xlabel("Tenure (months)"); plt.ylabel("Customers"); plt.legend(); plt.show()

## 4. Feature Engineering and Encoding

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No":0,"Yes":1})
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 5. Training and Testing

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
models={
"Logistic Regression":Pipeline([("preprocessor",preprocessor),("classifier",LogisticRegression(max_iter=2000,random_state=42))]),
"Random Forest":Pipeline([("preprocessor",preprocessor),("classifier",RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced"))])
}
for model in models.values(): model.fit(X_train,y_train)
print("Both models trained successfully.")

## 6. Model Evaluation

In [ ]:
results=[]
for name,model in models.items():
    pred=model.predict(X_test)
    results.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,zero_division=0),recall_score(y_test,pred,zero_division=0),f1_score(y_test,pred,zero_division=0)])
    print("\n",name); print(classification_report(y_test,pred,target_names=["No Churn","Churn"],zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_test,pred),display_labels=["No Churn","Churn"]).plot(); plt.title(name); plt.show()
results_df=pd.DataFrame(results,columns=["Model","Accuracy","Precision","Recall","F1-Score"])
display(results_df.round(4))

In [ ]:
results_df.set_index("Model")[["Accuracy","Precision","Recall","F1-Score"]].plot(kind="bar",ylim=(0,1))
plt.title("Model Performance Comparison"); plt.xticks(rotation=0); plt.show()
best_model_name=results_df.loc[results_df["F1-Score"].idxmax(),"Model"]
final_model=models[best_model_name]
print("Selected model:",best_model_name)

## 7. Application / Sample Prediction

In [ ]:
sample_customers=X.iloc[:2].copy()
pred=final_model.predict(sample_customers)
prob=final_model.predict_proba(sample_customers)[:,1]
output=sample_customers.copy()
output["Predicted Churn"]=np.where(pred==1,"Churn","No Churn")
output["Churn Probability"]=prob
display(output[["Predicted Churn","Churn Probability"]].round(3))

## Conclusion

The final model is selected using the higher test F1-score. The report should discuss the actual metrics, churn patterns, limitations, and how a telecom company could use predictions for retention analysis.